In [41]:
# Part 1: Pipeline Setup (Estimated: 10 minutes)
# Task: Configure Spark and define schemas
# etl_pipeline.py
"""
StreamPulse Real-Time ETL Pipeline
Kafka → Parse → Validate → Enrich → Window → Multi-Sink
"""
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    from_json, col, to_timestamp, when, lit, current_timestamp,
    window, count, avg, sum as _sum, min as _min, max as _max,
    countDistinct, hour, dayofweek, unix_timestamp, to_json, struct
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)

# ============================================
# Spark Session
# ============================================
spark = SparkSession.builder \
    .appName("StreamPulse-ETL") \
    .config("spark.sql.streaming.schemaInference", "true") \
    .config("spark.sql.shuffle.partitions", "6") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# ============================================
# Event Schema
# ============================================
EVENT_SCHEMA = StructType([
    StructField("event_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("action", StringType(), True),
    StructField("content_id", StringType(), True),
    StructField("show_id", StringType(), True),
    StructField("device", StringType(), True),
    StructField("country", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("duration_seconds", IntegerType(), True),
    StructField("session_id", StringType(), True),
])

VALID_ACTIONS = ['play', 'pause', 'complete', 'skip', 'like', 'share']


In [39]:
# Part 2: Ingestion Layer (Estimated: 10 minutes)
# Task: Read from Kafka and parse JSON
# ============================================
# Read from Kafka
# ============================================

# Define versions matching your Spark installation
raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "streaming.user.interactions") \
    .option("startingOffsets", "latest") \
    .option("maxOffsetsPerTrigger", 50000) \
    .option("failOnDataLoss", "false") \
    .load()

# ============================================
# Parse JSON
# ============================================
parsed = raw \
    .select(
        col("key").cast("string").alias("kafka_key"),
        from_json(col("value").cast("string"), EVENT_SCHEMA).alias("event"),
        col("timestamp").alias("kafka_ts"),
        col("partition").alias("kafka_partition"),
        col("offset").alias("kafka_offset"),
    ) \
    .select(
        "kafka_key", "event.*",
        "kafka_ts", "kafka_partition", "kafka_offset"
    )


AnalysisException: Failed to find data source: kafka. Please deploy the application as per the deployment section of Structured Streaming + Kafka Integration Guide.

In [ ]:
# Part 3: Validation Layer (Estimated: 15 minutes)
# Task: Validate events and route invalid ones to dead letter
# ============================================
# Validation: separate valid from invalid events
# ============================================
validated = parsed \
    .withColumn("event_time", to_timestamp(col("timestamp"))) \
    .withColumn(
        "is_valid",
        (col("user_id").isNotNull()) &
        (col("action").isin(VALID_ACTIONS)) &
        (col("event_time").isNotNull()) &
        (col("content_id").isNotNull())
    )

valid_events = validated.filter(col("is_valid") == True)
invalid_events = validated.filter(col("is_valid") == False)

# Write invalid events to dead letter topic
dead_letter_query = invalid_events \
    .select(
        col("kafka_key").alias("key"),
        to_json(struct("*")).alias("value")
    ) \
    .writeStream \
    .outputMode("append") \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("topic", "streaming.user.dead-letter") \
    .option("checkpointLocation", "/tmp/checkpoints/etl-dead-letter/") \
    .trigger(processingTime="30 seconds") \
    .start()


In [ ]:
## Part 4: Enrichment Layer (Estimated: 15 minutes)
# Task: Add business logic enrichments
# ============================================
# Enrich events with business logic
# ============================================
enriched = valid_events \
    .withColumn(
        "engagement_score",
        when(col("action") == "play", 1)
        .when(col("action") == "pause", 0)
        .when(col("action") == "complete", 3)
        .when(col("action") == "like", 5)
        .when(col("action") == "share", 7)
        .when(col("action") == "skip", -1)
        .otherwise(0)
    ) \
    .withColumn(
        "device_category",
        when(col("device").isin("mobile-ios", "mobile-android"), "mobile")
        .when(col("device") == "web", "web")
        .when(col("device").isin("smart-tv", "tv"), "tv")
        .otherwise("other")
    ) \
    .withColumn("hour_of_day", hour(col("event_time"))) \
    .withColumn("day_of_week", dayofweek(col("event_time"))) \
    .withColumn("processing_time", current_timestamp()) \
    .withColumn(
        "lateness_seconds",
        unix_timestamp(current_timestamp()) - unix_timestamp(col("event_time"))
    ) \
    .withColumn(
        "is_peak_hour",
        (hour(col("event_time")).between(18, 23))
    ) \
    .drop("is_valid")


In [ ]:
# Part 5: Data Lake Sink — Append Mode (Estimated: 15 minutes)
# Task: Write enriched events to Parquet
# ============================================
# Sink 1: Data Lake (Parquet) — raw enriched events
# ============================================
lake_query = enriched \
    .withColumn("event_date", col("event_time").cast("date")) \
    .writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", "/tmp/streampulse/lake/enriched-events/") \
    .option("checkpointLocation", "/tmp/checkpoints/etl-lake/") \
    .trigger(processingTime="1 minute") \
    .partitionBy("event_date", "country") \
    .start()

print("Data Lake sink started (Parquet, 1-minute trigger)")



In [ ]:
## Part 6: Windowed Metrics Sink — Update Mode (Estimated: 20 minutes)
# Task: Compute windowed aggregations with watermarks
# ============================================
# Sink 2: Windowed metrics (5-min tumbling windows)
# ============================================
windowed_metrics = enriched \
    .withWatermark("event_time", "10 minutes") \
    .groupBy(
        window(col("event_time"), "5 minutes"),
        col("action"),
        col("device_category")
    ) \
    .agg(
        count("*").alias("event_count"),
        countDistinct("user_id").alias("unique_users"),
        avg("engagement_score").alias("avg_engagement"),
        _sum("engagement_score").alias("total_engagement"),
        avg("duration_seconds").alias("avg_duration"),
        countDistinct("content_id").alias("unique_content"),
    )

metrics_query = windowed_metrics \
    .writeStream \
    .outputMode("update") \
    .format("console") \
    .option("truncate", False) \
    .trigger(processingTime="30 seconds") \
    .start()

print("Metrics sink started (console, 30-second trigger)")

# ============================================
# Sink 3: Show-level engagement (per 5-min window)
# ============================================
show_engagement = enriched \
    .withWatermark("event_time", "10 minutes") \
    .groupBy(
        window(col("event_time"), "5 minutes"),
        col("show_id")
    ) \
    .agg(
        count("*").alias("total_events"),
        countDistinct("user_id").alias("unique_viewers"),
        _sum("engagement_score").alias("engagement_total"),
        count(when(col("action") == "complete", 1)).alias("completions"),
        count(when(col("action") == "skip", 1)).alias("skips"),
    ) \
    .withColumn(
        "completion_rate",
        col("completions") / (col("completions") + col("skips"))
    )

show_query = show_engagement \
    .writeStream \
    .outputMode("update") \
    .format("console") \
    .option("truncate", False) \
    .queryName("show_engagement") \
    .trigger(processingTime="30 seconds") \
    .option("checkpointLocation", "/tmp/checkpoints/etl-show/") \
    .start()

print("Show engagement sink started")


In [ ]:
## Part 7: Pipeline Monitoring (Estimated: 10 minutes)
# Task: Monitor all running queries
# ============================================
# Monitor all queries
# ============================================
import time

def monitor_queries(duration=120):
    """Monitor all active streaming queries."""
    start = time.time()

    while time.time() - start < duration:
        print(f'\n{"="*60}')
        print(f'PIPELINE STATUS — {time.strftime("%H:%M:%S")}')
        print(f'{"="*60}')

        for query in spark.streams.active:
            status = query.status
            name = query.name or query.id

            is_active = '🟢' if status.get('isDataAvailable') else '⚪'

            print(f'\n  {is_active} Query: {name}')
            print(f'     Status: {status.get("message", "N/A")}')

            # Recent progress
            if query.recentProgress:
                latest = query.recentProgress[-1]
                print(f'     Input rows: {latest.get("numInputRows", 0)}')
                print(f'     Rows/sec: {latest.get("processedRowsPerSecond", 0):.0f}')
                print(f'     Batch: {latest.get("batchId", "N/A")}')

                # Watermark info
                if 'eventTime' in latest:
                    et = latest['eventTime']
                    print(f'     Watermark: {et.get("watermark", "N/A")}')

                # State info
                state_ops = latest.get('stateOperators', [])
                if state_ops:
                    print(f'     State rows: {state_ops[0].get("numRowsTotal", "N/A")}')

        time.sleep(15)

# Run monitoring for 2 minutes
monitor_queries(120)


In [ ]:
## Part 8: Verification (Estimated: 10 minutes)
# After running the pipeline for at least 5 minutes:
# Read back from data lake and verify
lake_data = spark.read.parquet("/tmp/streampulse/lake/enriched-events/")

print(f'Total events in lake: {lake_data.count()}')
print(f'Schema:')
lake_data.printSchema()

# Check enrichment
lake_data.groupBy("device_category").count().show()
lake_data.groupBy("engagement_score").count().show()

# Check partitioning
print(f'Countries:')
lake_data.select("country").distinct().show()

print(f'Event dates:')
lake_data.select("event_date").distinct().show()
